# Module 4 — Linked Lists

This is the worked reference notebook: run live in lecture, fully solved.
The version students receive with TODOs in place of the solved parts is
`assignments/pds/a3-linked-lists/starter/linked_lists.py`.

## 1. Singly linked list (Lecture 1)

In [1]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None

class LinkedList:
    def __init__(self):
        self.head = None
        self._size = 0

    def __len__(self):
        return self._size

    def is_empty(self):
        return self.head is None

    def push_front(self, value):
        new_node = Node(value)
        new_node.next = self.head
        self.head = new_node
        self._size += 1

    def push_back(self, value):
        new_node = Node(value)
        if self.head is None:
            self.head = new_node
            self._size += 1
            return
        current = self.head
        while current.next is not None:
            current = current.next
        current.next = new_node
        self._size += 1

    def to_list(self):
        result, current = [], self.head
        while current is not None:
            result.append(current.value)
            current = current.next
        return result

    def get(self, i):
        current = self.head
        for _ in range(i):
            current = current.next
        return current.value

    def reverse(self):
        prev, current = None, self.head
        while current is not None:
            nxt = current.next
            current.next = prev
            prev = current
            current = nxt
        self.head = prev

ll = LinkedList()
ll.push_back(10); ll.push_back(20); ll.push_back(30)
assert ll.to_list() == [10, 20, 30]
assert ll.get(1) == 20
ll.reverse()
assert ll.to_list() == [30, 20, 10]
print("Singly linked list checks passed")

Singly linked list checks passed


## 2. Doubly linked list (Lecture 2)

In [2]:
class DNode:
    def __init__(self, value):
        self.value = value
        self.next = None
        self.prev = None

class DoublyLinkedList:
    def __init__(self):
        self.head = None
        self.tail = None
        self._size = 0

    def __len__(self):
        return self._size

    def push_back_node(self, value):
        new_node = DNode(value)
        if self.tail is None:
            self.head = self.tail = new_node
        else:
            new_node.prev = self.tail
            self.tail.next = new_node
            self.tail = new_node
        self._size += 1
        return new_node

    def push_front(self, value):
        new_node = DNode(value)
        if self.head is None:
            self.head = self.tail = new_node
        else:
            new_node.next = self.head
            self.head.prev = new_node
            self.head = new_node
        self._size += 1
        return new_node

    def remove_node(self, node):
        if node.prev is not None:
            node.prev.next = node.next
        else:
            self.head = node.next
        if node.next is not None:
            node.next.prev = node.prev
        else:
            self.tail = node.prev
        self._size -= 1

    def to_list_forward(self):
        result, current = [], self.head
        while current is not None:
            result.append(current.value)
            current = current.next
        return result

dll = DoublyLinkedList()
n10 = dll.push_back_node(10)
n20 = dll.push_back_node(20)
n30 = dll.push_back_node(30)
assert dll.to_list_forward() == [10, 20, 30]
dll.remove_node(n20)   # O(1), given the reference directly
assert dll.to_list_forward() == [10, 30]
print("Doubly linked list checks passed")

Doubly linked list checks passed


## 3. Simplified LRU cache (Lecture 4, using the doubly linked list above)

In [3]:
class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.list = DoublyLinkedList()
        self.node_map = {}

    def access(self, key):
        if key in self.node_map:
            self.list.remove_node(self.node_map[key])
        new_node = self.list.push_back_node(key)
        self.node_map[key] = new_node
        if len(self.list) > self.capacity:
            evicted = self.list.head
            self.list.remove_node(evicted)
            del self.node_map[evicted.value]
        return self.list.to_list_forward()

cache = LRUCache(2)
assert cache.access('A') == ['A']
assert cache.access('B') == ['A', 'B']
assert cache.access('A') == ['B', 'A']
assert cache.access('C') == ['A', 'C']   # B evicted, matching Lecture 4's traced example
print("LRU cache checks passed, matching Lecture 4's worked trace")

LRU cache checks passed, matching Lecture 4's worked trace


## 4. Circular linked list and the Josephus problem (Lecture 3)

In [4]:
def josephus(n, k):
    head = Node(1)
    current = head
    for i in range(2, n + 1):
        current.next = Node(i)
        current = current.next
    current.next = head

    current = head
    while current.next is not current:
        for _ in range(k - 1):
            current = current.next
        current.next = current.next.next
    return current.value

assert josephus(5, 2) == 4    # matches Lecture 3's fully worked trace
assert josephus(6, 3) == 2    # matches Lecture 3's second worked trace
print("Josephus problem checks passed, matching Lecture 3's traces")

Josephus problem checks passed, matching Lecture 3's traces


## 5. Sparse polynomial (Lecture 4)

In [5]:
class Term:
    def __init__(self, coeff, exponent):
        self.coeff = coeff
        self.exponent = exponent
        self.next = None

class Polynomial:
    def __init__(self):
        self.head = None

    def add_term(self, coeff, exponent):
        new_term = Term(coeff, exponent)
        if self.head is None or exponent > self.head.exponent:
            new_term.next = self.head
            self.head = new_term
            return
        current = self.head
        while current.next and current.next.exponent > exponent:
            current = current.next
        new_term.next = current.next
        current.next = new_term

    def evaluate(self, x):
        total, current = 0, self.head
        while current is not None:
            total += current.coeff * (x ** current.exponent)
            current = current.next
        return total

    def terms(self):
        result, current = [], self.head
        while current is not None:
            result.append((current.coeff, current.exponent))
            current = current.next
        return result

p = Polynomial()
p.add_term(1, 1000)
p.add_term(3, 2)
p.add_term(1, 0)
assert p.terms() == [(1, 1000), (3, 2), (1, 0)]
assert p.evaluate(1) == 1 + 3 + 1   # 1^1000 + 3*1^2 + 1*1^0
assert p.evaluate(2) == 2**1000 + 3*4 + 1
print("Sparse polynomial checks passed")

Sparse polynomial checks passed
